In [0]:
from pyspark.sql import functions as F

CAMINHO = "/Volumes/voebem/bronze/arquivos/vra/*.csv"
TABELA = "voebem.bronze.vra"

In [0]:
bruto = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", True)
    .option("skipRows", 1)
    .option("quote", '"')
    .option("escape", '"')
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")
    .load(CAMINHO	)
)

print("colunas lidas do arquivo")
for c in bruto.columns:
  print(f" {c!r}")

colunas lidas do arquivo
 'ICAO Empresa Aérea'
 'Número Voo'
 'Código Autorização (DI)'
 'Código Tipo Linha'
 'ICAO Aeródromo Origem'
 'ICAO Aeródromo Destino'
 'Partida Prevista'
 'Partida Real'
 'Chegada Prevista'
 'Chegada Real'
 'Situação Voo'
 'Código Justificativa'


In [0]:
RENOMEAR = {
    "ICAO Empresa Aérea": "icao_empresa",
"Número Voo": "numero_voo",
"Código Autorização (DI)": "codigo_di",
"Código Tipo Linha": "codigo_tipo_linha",
"ICAO Aeródromo Origem": "icao_origem",
"ICAO Aeródromo Destino": "icao_destino",
"Partida Prevista": "partida_prevista",
"Partida Real": "partida_real",
"Chegada Prevista": "chegada_prevista",
"Chegada Real": "chegada_real",
"Situação Voo": "situacao_voo",
"Código Justificativa": "codigo_justificativa"
}

faltando = [c for c in RENOMEAR if c not in bruto.columns]
assert not faltando, f"Coluna esperada nao encontrada no CSV: {faltando}"

renomeado = bruto.select(
    *[F.col(f"`{origem}`").cast("string").alias(novo) for origem, novo in RENOMEAR.items()]
)

In [0]:
bronze = renomeado.withColumn(
    "_arquivo_origem", F.col("_metadata.file_name")
).withColumn(
    "_ingerido_em", F.current_timestamp()
)

In [0]:
(
    bronze.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA)
)

print(f"{TABELA}: {spark.table(TABELA).count():,} linhas")

voebem.bronze.vra: 1,014,705 linhas


In [0]:
spark.sql(f"""
COMMENT ON TABLE {TABELA} IS
'Bronze - VRA (Voo Regular Ativo) da ANAC, 12 meses (ago/2025 a jul/2026)
Dado bruto: todas as colunas string, nenhuma linha descartada. Carga full refresh idempotente a partir de /Volumes/voebem/bronze/arquivos/vra/'
""")

DataFrame[]

In [0]:
display(
    spark.sql(f"""
        SELECT _arquivo_origem, COUNT(*) AS Linhas, MAX(_ingerido_em) AS ingerido_em
        FROM {TABELA}
        GROUP BY _arquivo_origem
        ORDER BY _arquivo_origem
    """)
)

_arquivo_origem,Linhas,ingerido_em
VRA_202510.csv,85709,2026-09-17T16:57:57.913Z
VRA_202511.csv,82245,2026-09-17T16:57:57.913Z
VRA_202512.csv,88564,2026-09-17T16:57:57.913Z
VRA_20258.csv,84584,2026-09-17T16:57:57.913Z
VRA_20259.csv,82156,2026-09-17T16:57:57.913Z
VRA_20261.csv,91650,2026-09-17T16:57:57.913Z
VRA_20262.csv,79457,2026-09-17T16:57:57.913Z
VRA_20263.csv,87201,2026-09-17T16:57:57.913Z
VRA_20264.csv,80383,2026-09-17T16:57:57.913Z
VRA_20265.csv,82120,2026-09-17T16:57:57.913Z
